In [ ]:
from langgraph.graph import StateGraph,START,END
from pydantic import BaseModel
from typing import Literal
from IPython.display import Image, display

In [ ]:
class EmailState(BaseModel):
    email_content: str
    in_spam: bool = False
    classification: str = ""
    response: str = ""

In [ ]:
def check_spam(state: EmailState) -> EmailState:
    """Check whether the email is spam."""

    email = state.email_content.lower()

    spam_keywords = [
        "win money",
        "free prize",
        "click here",
        "congratulations",
        "lottery",
        "you have won",
    ]

    is_spam = any(keyword in email for keyword in spam_keywords)

    return {"in_spam": is_spam}

In [ ]:
def classify_email(state: EmailState) -> EmailState:
    """Classify the email based on its content."""

    if state.in_spam:
        classification = "spam"
    else:
        email = state.email_content.lower()

        if any(word in email for word in ["invoice", "payment", "bill"]):
            classification = "finance"

        elif any(word in email for word in ["meeting", "schedule", "appointment"]):
            classification = "meeting"

        elif any(word in email for word in ["bug", "error", "technical", "login"]):
            classification = "technical"

        elif any(word in email for word in ["job", "application", "resume", "cv"]):
            classification = "career"

        else:
            classification = "general"

    return {"classification": classification}




In [ ]:
def generate_response(state: EmailState) -> EmailState:
    """Generate the final response."""

    if state.in_spam:
        response = "This email has been identified as spam and will not be processed."

    elif state.classification == "finance":
        response = (
            "Thank you for contacting us regarding your financial inquiry. "
            "Our finance team will review your request and get back to you."
        )

    elif state.classification == "meeting":
        response = (
            "Thank you for your email. We have received your meeting request "
            "and will confirm the schedule shortly."
        )

    elif state.classification == "technical":
        response = (
            "Thank you for contacting technical support. "
            "We have received your issue and will investigate it shortly."
        )

    elif state.classification == "career":
        response = (
            "Thank you for your interest in our company. "
            "Your application has been received and will be reviewed by our team."
        )

    else:
        response = (
            "Thank you for your email. "
            "We have received your message and will get back to you soon."
        )

    return {"response": response}


In [ ]:
builder=StateGraph(EmailState)

In [ ]:
builder.add_node("check_spam",check_spam)
builder.add_node("classify_email",classify_email)
builder.add_node("generate_response",generate_response)

In [ ]:
builder.add_edge(START,"check_spam")
builder.add_edge("check_spam","classify_email")
builder.add_edge("classify_email","generate_response")
builder.add_edge("generate_response",END)

In [ ]:
graph=builder.compile()

In [ ]:
Image(graph.get_graph().draw_mermaid_png())

In [ ]:
initial_state=EmailState(email_content="Hello, I am a spam email. Please do not process this email.")

In [ ]:
result=graph.invoke(initial_state)

In [ ]:
print(result)